In [2]:
%load_ext autoreload
%autoreload 2
import importlib
import journey_analysis

importlib.reload(journey_analysis)

from journey_analysis import build_sankey

from build_site import build_summit_outfitters
from visitor import Visitor
from Behaviour import all_behavior_rules
from simulation import simulate_visitors

from journey_analysis import (
    build_event_table,
    build_transition_table,
    summarize_transitions,
    summarize_transitions_by_page_type,
    build_sankey,
    summarize_transitions_for_sankey,
    summarize_transitions_by_step,
    build_step_sankey
)

In [3]:
import inspect
print(inspect.signature(build_sankey))

(transition_summary, source_col='page_id', target_col='next_page_id')


In [4]:
site = build_summit_outfitters()

In [5]:
type_transition_summary = summarize_transitions_by_page_type(
    transitions,
    site
)

NameError: name 'transitions' is not defined

In [ ]:
visitor = Visitor(
    visitor_id="V000001",
    user_type="researcher",
    buyer_intent="high"
)

In [ ]:
journeys = simulate_visitors(
    site=site,
    rules=all_behavior_rules,
    number_of_visitors=1000,
    seed=42
)

In [ ]:
events = build_event_table(journeys)

In [ ]:
events.head(20)

,visitor_id,user_type,buyer_intent,step,page_id,converted,end_reason
0,V000001,researcher,low,0,home,False,exit
1,V000001,researcher,low,1,category_footwear,False,exit
2,V000001,researcher,low,2,product_footwear_05,False,exit
3,V000001,researcher,low,3,shipping_info,False,exit
4,V000001,researcher,low,4,exit,False,exit
5,V000002,browser,low,0,home,False,exit
6,V000002,browser,low,1,category_mens_clothing,False,exit
7,V000002,browser,low,2,product_mens_clothing_03,False,exit
8,V000002,browser,low,3,shipping_info,False,exit
9,V000002,browser,low,4,exit,False,exit


In [ ]:
journeys["end_reason"].value_counts()

end_reason
exit            581
confirmation    418
max_steps         1
Name: count, dtype: int64

In [ ]:
journeys["user_type"].value_counts(normalize=True)

user_type
browser           0.335
researcher        0.334
deal_seeker       0.196
direct_shopper    0.135
Name: proportion, dtype: float64

In [ ]:
journeys["buyer_intent"].value_counts(normalize=True)

buyer_intent
medium    0.381
high      0.334
low       0.285
Name: proportion, dtype: float64

In [ ]:
journeys["end_reason"].value_counts()

end_reason
exit            581
confirmation    418
max_steps         1
Name: count, dtype: int64

In [ ]:
journeys["converted"].mean()

np.float64(0.418)

In [ ]:
(
    journeys.loc[
        journeys["end_reason"] == "no_available_transition",
        "journey"
    ]
    .apply(lambda x: x[-1])
    .value_counts()
)

Series([], Name: count, dtype: int64)

In [ ]:
site.pages[
    site.pages["page_id"].isin([
        "recommendations",
        "product_help",
        "shipping_info",
        "login",
        "store_locator",
        "about"
    ])
][["page_id", "page_type"]].to_dict("records")

[{'page_id': 'recommendations', 'page_type': 'recommendation'},
 {'page_id': 'login', 'page_type': 'account'},
 {'page_id': 'shipping_info', 'page_type': 'support'},
 {'page_id': 'product_help', 'page_type': 'support'},
 {'page_id': 'about', 'page_type': 'corporate'},
 {'page_id': 'store_locator', 'page_type': 'corporate'}]

In [ ]:
site.edges[
    site.edges["source"] == "recommendations"
]

,source,target,link_type
92,recommendations,home,main_navigation
93,recommendations,exit,exit
94,recommendations,product_footwear_01,recommended_product
95,recommendations,product_mens_clothing_01,recommended_product
96,recommendations,product_womens_clothing_01,recommended_product
97,recommendations,product_camping_01,recommended_product
98,recommendations,product_hiking_01,recommended_product
99,recommendations,product_accessories_01,recommended_product


In [ ]:
transitions = build_transition_table(events)

In [ ]:
type_transition_summary = summarize_transitions_by_page_type(
    transitions,
    site
)

In [ ]:
fig.update_layout(
    title_text="Summit Outfitters Visitor Journeys by Page Type",
    font_size=14,
    width=1000,
    height=700
)

In [ ]:
fig = build_sankey(
    sankey_summary,
    source_col="source_node",
    target_col="target_node"
)

fig.show()

In [ ]:
type_transition_summary.head(15)

,page_type,next_page_type,count
0,checkout,checkout,836
1,category,product,769
2,product,product,598
3,home,category,516
4,cart,checkout,418
5,checkout,confirmation,418
6,product,cart,380
7,search,search_results,365
8,product,exit,302
9,search_results,product,252


In [ ]:
sankey_summary = summarize_transitions_for_sankey(
    transitions,
    site
)



In [ ]:
sankey_summary.head(15)

,source_node,target_node,count
0,category,product,769
1,product,product,598
2,home,category,516
3,cart,checkout_shipping,418
4,checkout_shipping,checkout_payment,418
5,checkout_payment,checkout_review,418
6,checkout_review,confirmation,418
7,product,cart,380
8,search,search_results,365
9,product,exit,302


In [ ]:
sankey_summary[
    (sankey_summary["source_node"] == "checkout_shipping") &
    (sankey_summary["target_node"] == "confirmation")
]

,source_node,target_node,count


In [ ]:
site.edges[
    site.edges["source"] == "checkout_shipping"
]

,source,target,link_type
63,checkout_shipping,checkout_payment,checkout


In [ ]:
fig = build_sankey(
    sankey_summary,
    source_col="source_node",
    target_col="target_node"
)

fig.show()

In [ ]:
fig = build_sankey(
    sankey_summary,
    source_col="source_node",
    target_col="target_node"
)

fig.show()

In [ ]:
step_transition_summary = summarize_transitions_by_step(
    events,
    site
)

In [ ]:
step_transition_summary.head(25)

,source_node,target_node,count
0,Step 0: home,Step 1: category,433
1,Step 1: category,Step 2: product,433
2,Step 2: product,Step 3: product,187
3,Step 0: home,Step 1: search,156
4,Step 1: search,Step 2: search_results,127
5,Step 0: home,Step 1: sale,124
6,Step 3: category,Step 4: product,122
7,Step 5: checkout_payment,Step 6: checkout_review,116
8,Step 4: checkout_shipping,Step 5: checkout_payment,116
9,Step 6: checkout_review,Step 7: confirmation,116


In [ ]:
fig = build_sankey(
    step_transition_summary,
    source_col="source_node",
    target_col="target_node"
)

fig.show()

In [ ]:
fig = build_step_sankey(
    step_transition_summary,
    max_step=7
)

fig.show()

In [ ]:
step_transition_summary_filtered = step_transition_summary[
    step_transition_summary["count"] >= 10
].copy()

In [ ]:
fig = build_step_sankey(
    step_transition_summary_filtered,
    max_step=7
)

fig.show()

In [ ]:
fig = build_step_sankey(
    step_transition_summary,
    max_step=7,
    min_count=10
)

fig.show()

In [ ]:
search_transitions = transitions[
    transitions["page_id"] == "search"
]

search_transitions.groupby(
    ["converted", "next_page_id"]
).size()

NameError: name 'transitions' is not defined